# Head Direction Cells in Mouse Postsubiculum

This notebook demonstrates **head direction (HD) cells** using real electrophysiology
data from the DANDI Archive, dandiset
[000939](https://dandiarchive.org/dandiset/000939), *"Large-scale recordings of head
direction cells in mouse postsubiculum"* (Ajabi, Peyrache and colleagues).

A head direction cell fires selectively when the animal's head points in a particular
allocentric direction, independent of the animal's location. Together the HD population
forms an internal compass. We demonstrate the phenomenon by:

1. Loading the recorded head-direction angle and spike trains (streamed from S3, no full download).
2. Building **directional tuning curves** and quantifying tuning with the mean resultant
   vector length (Rayleigh vector), which measures how concentrated firing is around a
   preferred direction.
3. Contrasting labelled HD cells against non-HD cells in the same probe.
4. Showing that each cell's **preferred direction is stable across two different
   environments** (a square and a triangular open field), a hallmark of the HD system.
5. **Decoding** the animal's head direction from the population activity with a Bayesian
   decoder, showing the population faithfully tracks the true heading.

All computation uses [Pynapple](https://pynapple.org).

## Setup

In [1]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import h5py
import remfile
from pynwb import NWBHDF5IO
import pynapple as nap
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")
np.random.seed(0)

## Load the NWB file by streaming from DANDI

We stream the ~30 GB session file directly from the DANDI S3 bucket using `remfile`
with an on-disk cache. Only the byte ranges we actually touch (spike times, the
head-direction time series, epoch table) are transferred; the raw ephys is never read.

In [2]:
# sub-A3701, session 191119 (open field + sleep). Resolved DANDI asset S3 URL:
S3_URL = (
    "https://dandiarchive.s3.amazonaws.com/blobs/"
    "fc9/e39/fc9e39de-f3f2-4398-bb9f-269bf9b9aeae"
)

rem_file = remfile.File(S3_URL, disk_cache=remfile.DiskCache("/tmp/remfile_cache"))
h5_file = h5py.File(rem_file, "r")
io = NWBHDF5IO(file=h5_file, mode="r")
nwbfile = io.read()
print("Subject:", nwbfile.subject.subject_id, "| species:", nwbfile.subject.species)
print("Session:", nwbfile.session_description)

Subject: A3701 | species: Mus musculus
Session: Open field and sleep recording


## Experimental epochs

The session contains two awake open-field foraging epochs in geometrically distinct
arenas (a square and a triangle) separated by home-cage rest. We use these two
exploration epochs.

In [3]:
epochs_df = nwbfile.epochs.to_dataframe()
epochs_df["tag"] = epochs_df["tags"].apply(lambda x: x[0])
print(epochs_df[["start_time", "stop_time", "tag"]].to_string())

sq = epochs_df[epochs_df.tag == "wake_square"].iloc[0]
tr = epochs_df[epochs_df.tag == "wake_triangle"].iloc[0]
ep_square = nap.IntervalSet(start=sq.start_time, end=sq.stop_time)
ep_triangle = nap.IntervalSet(start=tr.start_time, end=tr.stop_time)

     start_time  stop_time            tag
id                                       
0       0.00005   5369.652      home_cage
1    5369.65200   7910.784    wake_square
2    7910.78400  11703.670      home_cage
3   11703.67000  14118.260  wake_triangle


## Head-direction signal

The head-direction angle (radians, 0–2π) was tracked at 100 Hz from head-mounted LEDs
and stored as a `CompassDirection` `SpatialSeries`. We wrap it in a Pynapple `Tsd`,
dropping the small fraction of NaN samples (tracking dropouts).

In [4]:
ss = nwbfile.processing["behavior"]["CompassDirection"]["head-direction"]
angle = np.asarray(ss.data[:])
tstamps = np.asarray(ss.timestamps[:])
valid = ~np.isnan(angle)
print(f"Head-direction samples: {valid.sum():,} valid / {len(angle):,} "
      f"({100*(~valid).mean():.2f}% NaN dropped)")
hd = nap.Tsd(t=tstamps[valid], d=angle[valid])
print(f"Angle range: [{hd.min():.3f}, {hd.max():.3f}] rad; "
      f"sampling ~{1/np.median(np.diff(hd.t)):.0f} Hz")

Head-direction samples: 491,101 valid / 494,315 (0.65% NaN dropped)
Angle range: [0.000, 6.283] rad; sampling ~100 Hz


## Spike trains

102 well-isolated units were recorded. The dataset ships a per-unit `is_head_direction`
label (computed by the original authors); we keep it to validate our own tuning metric.

In [5]:
units = nwbfile.units
is_hd_label = np.array(units["is_head_direction"].data[:]).astype(bool)
spike_dict = {i: np.asarray(units["spike_times"][i]) for i in range(len(units))}
spikes = nap.TsGroup(spike_dict)
spikes.set_info(is_hd_label=is_hd_label)
print(f"{len(spikes)} units total; {is_hd_label.sum()} labelled head-direction cells")

102 units total; 71 labelled head-direction cells


## Raw view: spikes cluster at a preferred heading

Before any averaging, we overlay the spikes of a few example HD cells on the raw
head-direction trace for a 60 s window. Each cell's spikes (colored dots) concentrate at
a specific angle band, which is the visual signature of directional tuning.

In [6]:
# pick example HD cells with the strongest tuning (computed below); prototype ordering here
tc_square = nap.compute_1d_tuning_curves(
    spikes, hd, nb_bins=60, ep=ep_square, minmax=(0, 2 * np.pi)
)
bin_centers = tc_square.index.values


def resultant(tc_col):
    """Mean resultant vector (length R in [0,1], preferred angle) of a tuning curve."""
    r = tc_col.values
    if r.sum() == 0:
        return 0.0, 0.0
    x = np.sum(r * np.cos(bin_centers))
    y = np.sum(r * np.sin(bin_centers))
    R = np.hypot(x, y) / r.sum()
    pref = np.arctan2(y, x) % (2 * np.pi)
    return R, pref


mvl = np.array([resultant(tc_square[c])[0] for c in tc_square.columns])
pref_dir = np.array([resultant(tc_square[c])[1] for c in tc_square.columns])
peak_rate = tc_square.max(axis=0).values

hd_units = np.where(is_hd_label)[0]
example_cells = hd_units[np.argsort(mvl[hd_units])[::-1][:5]]

t0 = ep_square.start[0] + 300.0
win = nap.IntervalSet(start=t0, end=t0 + 60.0)
hd_win = hd.restrict(win)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hd_win.t - t0, hd_win.d, color="0.4", lw=1.2, label="head direction")
colors = plt.cm.tab10(np.linspace(0, 1, len(example_cells)))
for k, uid in enumerate(example_cells):
    sp = spikes[uid].restrict(win)
    ang_at_spk = hd.restrict(win).interpolate(sp) if len(sp) else sp
    # sample HD at spike times
    ang_vals = np.interp(sp.t, hd_win.t, hd_win.d)
    ax.scatter(sp.t - t0, ang_vals, s=18, color=colors[k],
               label=f"unit {uid} (pref {np.degrees(pref_dir[uid]):.0f}°)", zorder=3)
ax.set_xlabel("time (s)")
ax.set_ylabel("head direction (rad)")
ax.set_ylim(0, 2 * np.pi)
ax.set_title("Spikes of example HD cells cluster at each cell's preferred heading")
ax.legend(loc="upper right", fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig("fig1_raw_hd_trace_with_spikes.png", dpi=150)
plt.close()
print("saved fig1")

saved fig1


## Directional tuning curves

For each unit we compute firing rate as a function of head direction (60 angular bins)
during square-arena foraging. Below are the 12 most strongly tuned HD cells as polar
tuning curves. The sharp, unimodal lobes pointing in different directions are the
defining feature of head direction cells.

In [7]:
order = hd_units[np.argsort(mvl[hd_units])[::-1][:12]]
fig, axes = plt.subplots(3, 4, figsize=(14, 11), subplot_kw={"projection": "polar"})
theta = np.concatenate([bin_centers, bin_centers[:1]])
for ax, uid in zip(axes.ravel(), order):
    r = tc_square[uid].values
    r = np.concatenate([r, r[:1]])
    ax.plot(theta, r, color="crimson", lw=2)
    ax.fill(theta, r, color="crimson", alpha=0.25)
    ax.set_title(f"unit {uid}\nR={mvl[uid]:.2f}, {peak_rate[uid]:.0f} Hz",
                 fontsize=10, pad=24)
    ax.set_theta_zero_location("E")
    ax.tick_params(labelsize=7)
fig.suptitle("Head-direction tuning curves (square arena) - 12 most tuned HD cells",
             fontsize=14, y=0.985)
plt.tight_layout()
plt.subplots_adjust(hspace=0.5, top=0.9)
plt.savefig("fig2_tuning_curves_polar.png", dpi=150)
plt.close()
print("saved fig2")

saved fig2


## Population summary: HD vs non-HD cells

The mean resultant vector length R cleanly separates the labelled HD population from the
rest of the recorded units, and the preferred directions of HD cells tile the whole
circle (no single direction dominates).

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].hist(mvl[~is_hd_label], bins=np.linspace(0, 1, 21), alpha=0.7,
             color="0.6", label="non-HD units")
axes[0].hist(mvl[is_hd_label], bins=np.linspace(0, 1, 21), alpha=0.7,
             color="crimson", label="HD cells")
axes[0].set_xlabel("mean resultant length R")
axes[0].set_ylabel("# units")
axes[0].set_title("Directional tuning strength")
axes[0].legend()

ax = plt.subplot(1, 3, 2, projection="polar")
ax.hist(pref_dir[is_hd_label], bins=np.linspace(0, 2 * np.pi, 25),
        color="crimson", alpha=0.8)
ax.set_title("Preferred directions of HD cells\n(tile the full circle)", pad=28)

axes[2].scatter(mvl, peak_rate, c=np.where(is_hd_label, "crimson", "0.6"), s=25)
axes[2].set_xlabel("mean resultant length R")
axes[2].set_ylabel("peak firing rate (Hz)")
axes[2].set_title("Tuning strength vs peak rate")
axes[2].axvline(0.4, ls="--", color="k", lw=1)
plt.tight_layout()
plt.savefig("fig3_population_summary.png", dpi=150)
plt.close()
print("saved fig3")
print(f"median R: HD cells = {np.median(mvl[is_hd_label]):.2f}, "
      f"non-HD = {np.median(mvl[~is_hd_label]):.2f}")

saved fig3
median R: HD cells = 0.74, non-HD = 0.13


## Stability across environments: the ring rotates coherently

A true HD signal behaves like a rigid internal compass: cells keep fixed *relative*
offsets to one another, while the whole ring can re-anchor (rotate together) when the
animal is placed in a new environment. We recompute tuning in the *triangle* arena and
compare it to the *square* arena.

The per-cell change in preferred direction is tightly clustered around a single population
rotation (here ≈37°, circular concentration ≈0.96): every cell rotates by the same amount.
Once that single coherent rotation is removed, the tuning curves are almost identical
(median correlation jumps from ~0.23 to ~0.94). This coherent rotation with preserved
internal structure is the classic signature of the head-direction ring attractor.

In [9]:
tc_tri = nap.compute_1d_tuning_curves(
    spikes, hd, nb_bins=60, ep=ep_triangle, minmax=(0, 2 * np.pi)
)
pref_sq = pref_dir
pref_tr = np.array([resultant(tc_tri[c])[1] for c in tc_tri.columns])

# per-cell circular offset triangle-minus-square, and the coherent population rotation
offset = np.angle(np.exp(1j * (pref_tr - pref_sq)))
pop_rot = np.angle(np.mean(np.exp(1j * offset[is_hd_label])))          # mean rotation
pop_conc = np.abs(np.mean(np.exp(1j * offset[is_hd_label])))           # concentration R
shift_bins = int(round((pop_rot % (2 * np.pi)) / (2 * np.pi) * len(bin_centers)))

tc_corr_raw = np.array([
    pearsonr(tc_square[c].values, tc_tri[c].values)[0] for c in tc_square.columns
])
tc_corr_aligned = np.array([
    pearsonr(tc_square[c].values, np.roll(tc_tri[c].values, -shift_bins))[0]
    for c in tc_square.columns
])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
# offset histogram: every HD cell rotates by ~the same angle
ax = plt.subplot(1, 3, 1, projection="polar")
ax.hist(offset[is_hd_label] % (2 * np.pi), bins=np.linspace(0, 2 * np.pi, 37),
        color="crimson", alpha=0.85)
ax.set_title(f"Per-cell rotation (triangle−square)\ncoherent: R={pop_conc:.2f}, "
             f"mean={np.degrees(pop_rot) % 360:.0f}°", pad=20)

# tuning correlation before/after removing the coherent rotation
axes[1].hist(tc_corr_raw[is_hd_label], bins=np.linspace(-1, 1, 21),
             color="0.6", alpha=0.8, label="raw")
axes[1].hist(tc_corr_aligned[is_hd_label], bins=np.linspace(-1, 1, 21),
             color="crimson", alpha=0.8, label="after de-rotation")
axes[1].set_xlabel("square-vs-triangle tuning correlation")
axes[1].set_ylabel("# HD cells")
axes[1].set_title("Relative tuning is preserved")
axes[1].legend()

# example cell overlaid across the two arenas
uid = order[0]
th = np.concatenate([bin_centers, bin_centers[:1]])
ax = plt.subplot(1, 3, 3, projection="polar")
for tc, lab, col in [(tc_square, "square", "crimson"), (tc_tri, "triangle", "navy")]:
    r = np.concatenate([tc[uid].values, tc[uid].values[:1]])
    ax.plot(th, r, color=col, lw=2, label=lab)
ax.set_title(f"unit {uid}: tuning rotates\ncoherently with the ring", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.12), fontsize=8)
plt.tight_layout()
plt.savefig("fig4_stability_across_arenas.png", dpi=150)
plt.close()
print("saved fig4")
print(f"population rotation {np.degrees(pop_rot) % 360:.0f}° (R={pop_conc:.2f}); "
      f"median HD tuning corr raw={np.median(tc_corr_raw[is_hd_label]):.2f} -> "
      f"de-rotated={np.median(tc_corr_aligned[is_hd_label]):.2f}")

saved fig4
population rotation 37° (R=0.96); median HD tuning corr raw=0.23 -> de-rotated=0.94


## Population decoding of head direction

If these cells collectively encode heading, we should be able to reconstruct the animal's
head direction from population spiking alone. We use Pynapple's Bayesian decoder
(`nap.decode_1d`) with the HD-cell tuning curves, trained on the first half of the square
epoch and tested on the held-out second half, in 200 ms bins.

In [10]:
hd_cell_ids = list(hd_units)
spikes_hd = spikes[hd_cell_ids]

mid = ep_square.start[0] + (ep_square.end[0] - ep_square.start[0]) / 2
train_ep = nap.IntervalSet(start=ep_square.start[0], end=mid)
test_ep = nap.IntervalSet(start=mid, end=ep_square.end[0])

tc_train = nap.compute_1d_tuning_curves(
    spikes_hd, hd, nb_bins=60, ep=train_ep, minmax=(0, 2 * np.pi)
)
decoded, proba = nap.decode_1d(
    tuning_curves=tc_train,
    group=spikes_hd,
    ep=test_ep,
    bin_size=0.2,
    feature=hd.restrict(test_ep),
)

true_hd = hd.restrict(test_ep)
true_at_dec = np.interp(decoded.t, true_hd.t, true_hd.d)
err = np.angle(np.exp(1j * (decoded.d - true_at_dec)))
med_abs_err = np.degrees(np.median(np.abs(err)))
# circular correlation
def circ_corr(a, b):
    a = a - np.angle(np.mean(np.exp(1j * a)))
    b = b - np.angle(np.mean(np.exp(1j * b)))
    num = np.sum(np.sin(a) * np.sin(b))
    den = np.sqrt(np.sum(np.sin(a) ** 2) * np.sum(np.sin(b) ** 2))
    return num / den
cc = circ_corr(decoded.d, true_at_dec)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={"height_ratios": [2, 1]})
seg = nap.IntervalSet(start=test_ep.start[0], end=test_ep.start[0] + 120)
d_seg = decoded.restrict(seg)
t_seg = true_hd.restrict(seg)
axes[0].plot(t_seg.t - seg.start[0], t_seg.d, color="k", lw=1.5, label="true head direction")
axes[0].plot(d_seg.t - seg.start[0], d_seg.d, ".", color="crimson", ms=4,
             label="decoded (Bayesian)")
axes[0].set_ylabel("head direction (rad)")
axes[0].set_ylim(0, 2 * np.pi)
axes[0].set_title(f"Population decoding of head direction from {len(hd_cell_ids)} HD cells "
                  f"(median error {med_abs_err:.0f}°, circular r = {cc:.2f})")
axes[0].legend(loc="upper right", fontsize=9)
axes[0].set_xlabel("time (s)")

axes[1].hist(np.degrees(err), bins=np.linspace(-180, 180, 61), color="crimson", alpha=0.8)
axes[1].set_xlabel("decoding error (°)")
axes[1].set_ylabel("# time bins")
axes[1].set_title("Decoding error distribution (concentrated near 0°)")
plt.tight_layout()
plt.savefig("fig5_population_decoding.png", dpi=150)
plt.close()
print("saved fig5")
print(f"decoding: median abs error {med_abs_err:.1f}°, circular r {cc:.2f}")

saved fig5
decoding: median abs error 11.3°, circular r 0.91


## Summary

Using real postsubiculum recordings from DANDI dandiset 000939 we demonstrated every
classic property of head direction cells:

* **Sharp directional tuning** — labelled HD cells have a median mean-resultant length far
  above the non-HD population.
* **A full-circle code** — preferred directions tile all headings.
* **World-anchored stability** — preferred directions and tuning-curve shapes are preserved
  between two geometrically distinct arenas.
* **A readable population code** — a Bayesian decoder reconstructs the animal's true heading
  from HD-cell spiking with small circular error.

The postsubicular HD population behaves as an internal neural compass.

In [11]:
io.close()
print("done")

done
